## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI #LLM abstraction
import os 
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

e:\python 314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from openai import OpenAI
DB_NAME = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_url = "https://openrouter.ai/api/v1"
MODEL = "gpt-oss-20b:free"
openai = OpenAI(base_url=openrouter_url,api_key=openai_api_key)

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [5]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL,openai_api_base=openrouter_url,openai_api_key=openai_api_key)

### These LangChain objects implement the method `invoke()`

In [6]:
retriever.invoke("Who is Avery?")

[Document(id='de205c31-20e6-46fb-b418-cb318e048291', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [7]:
llm.invoke("Who is Avery?")

AIMessage(content='I’m not sure which Avery you’re referring to—there are many people, characters, and even companies with that name. Could you let me know a bit more about the context? For example, are you asking about a public figure, a fictional character, a historical person, or something else entirely?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 182, 'prompt_tokens': 71, 'total_tokens': 253, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 111, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.37695e-05, 'upstream_inference_prompt_cost': 1.0295e-06, 'upstream_inference_completions_cost': 1.274e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-20b:free', 'system_fingerprint'

## Time to put this together!

In [8]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [9]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [10]:
answer_question("Who is Averi Lancaster?", [])

'**Avery Lancaster** (sometimes misspelled as “Averi”) is the Co‑Founder and Chief Executive Officer (CEO) of Insurellm.  \n\n- **Born:** March\u202f15\u202f1985  \n- **Location:** San\u202fFrancisco, California  \n- **Current Salary:** $225,000  \n\n**Career Highlights at Insurellm**  \n- **2015‑Present:** Co‑Founder & CEO – led the company from its founding to becoming a leading provider in the Insurance‑Tech space.  \n- **2013‑2015:** Senior Product Manager at Innovate Insurance Solutions, where she created innovative tech‑focused insurance products.  \n- **2021‑Present:** Senior Data Engineer – promoted after driving a project that cut data retrieval times by 30%, and she now mentors junior engineers and shapes strategic data initiatives.\n\nAvery is known for her innovative leadership, risk‑management expertise, and her ability to steer Insurellm into mainstream insurance markets.'

## What could possibly come next? 😂

In [11]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!